<a href="https://colab.research.google.com/github/yiii1014/114-2-/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1Moh2bKPmrs-PWC_C_7h9F5JAatitPm7LV171nM7ilz8/edit?usp=sharing


In [1]:
import gradio as gr
import pandas as pd
import datetime
import gspread
from google.colab import auth
from google.auth import default
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1Moh2bKPmrs-PWC_C_7h9F5JAatitPm7LV171nM7ilz8/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"

REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人", "地點", "支付方式"]

_auth_done = False
_gc = None
_ws = None

In [3]:
#def add_expense(date_str, time_str, category, item, amount, payer):

In [4]:
def _ensure_headers():
    """確保表頭包含 REQUIRED_COLUMNS；若空表或缺欄，會補齊。"""
    rows = _ws.get_all_values()
    if not rows:
        _ws.append_row(REQUIRED_COLUMNS, value_input_option="USER_ENTERED")
        return
    header = rows[0]
    print(f"目前的標頭欄位為: {header}")
    if header != REQUIRED_COLUMNS:
        # 合併既有欄與必需欄，並以 REQUIRED_COLUMNS 為主順序
        existing = {h: idx for idx, h in enumerate(header)}
        # 若第一列不是必需欄，重寫表頭並搬移資料欄位（簡化處理：補欄位）
        _ws.update('1:1', [REQUIRED_COLUMNS])

        # 若舊資料有相同欄位名，保留；沒有的欄位留空
        if len(rows) > 1:
            new_rows = []
            for r in rows[1:]:
                mapping = {col: (r[existing[col]] if col in existing and existing[col] < len(r) else "")
                           for col in REQUIRED_COLUMNS}
                new_rows.append([mapping[c] for c in REQUIRED_COLUMNS])
            # 先清掉舊內容只留表頭
            _ws.resize(rows=1)
            # 再補回資料
            _ws.append_rows(new_rows, value_input_option="USER_ENTERED")

In [5]:
def _ensure_auth_and_ws():
    global _auth_done, _gc, _ws
    if not _auth_done:
        # Colab 使用者授權
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _auth_done = True
    if _ws is None:
        gs = _gc.open_by_url(SHEET_URL)
        _ws = gs.worksheet(WORKSHEET_NAME)
        # 確保欄位完整
        _ensure_headers()
    return _ws

In [6]:
sheets = _ensure_auth_and_ws().get_all_values()
df = pd.DataFrame(sheets[1:], columns=sheets[0])
df

目前的標頭欄位為: ['日期', '時間', '分類', '品項', '金額', '付款人', '地點', '支付方式', '備註']


/tmp/ipykernel_293/839643050.py:13: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  _ws.update('1:1', [REQUIRED_COLUMNS])


,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
0,2026-03-05,10:01,早餐,豆漿,30,yiii,,,
1,2026-03-12,10:40,外食,咖啡,30,yiii,,,
2,2026-03-12,10:40,交通,公車,15,yiii,,,
3,2026-03-12,11:20,外食,便當,100,yiii,,,
4,2026-03-14,7:14,外食,關東煮,150,yiii,公館夜市,現金,


In [7]:
def add_expense(date_str, time_str, category, item, amount, payer, location, payment_method):
    try:
        # 基本驗證
        if not date_str:
            date_str = datetime.date.today().strftime("%Y-%m-%d")
        # 允許空白時間
        if time_str:
            datetime.datetime.strptime(time_str, "%H:%M")
        # 類別/品項/付款人預設
        category = (category or "未填").strip()
        item = (item or "未填").strip()
        payer = (payer or "匿名").strip()
        location = (location or "").strip()
        payment_method = (payment_method or "").strip()

        # 金額
        try:
            amount_val = float(amount)
        except:
            return "金額格式錯誤，請輸入數字", None, None, None

        ws = _ensure_auth_and_ws()
        # 直接 append 一列
        ws.append_row(
            [date_str, time_str or "", category, item, amount_val, payer, location, payment_method],
            value_input_option="USER_ENTERED"
        )
        msg = f"✅ 已新增：{date_str} {time_str or ''}｜{category}｜{item}｜{amount_val}｜{payer}｜{location}｜{payment_method}"
        # 回傳即時摘要
        df = _read_df()
        cat, settle = _make_summary_tables(df)
        total = float(df["金額"].sum()) if not df.empty else 0.0
        return msg, total, cat, settle
    except Exception as e:
        return f"❌ 新增失敗：{e}", None, None, None

In [8]:
def _read_df():
    ws = _ensure_auth_and_ws()
    values = ws.get_all_values()
    if not values:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)
    df = pd.DataFrame(values[1:], columns=values[0])
    # 型別整理
    if "金額" in df.columns:
        df["金額"] = pd.to_numeric(df["金額"], errors="coerce").fillna(0.0)
    if "日期" in df.columns:
        df["日期"] = pd.to_datetime(df["日期"], errors="coerce")
        df["月份"] = df["日期"].dt.to_period('M') # Add month column for grouping
    # Ensure '分類' has a default for empty strings
    if "分類" in df.columns:
        # Fill NaN values with empty string first, then replace empty/whitespace strings
        df["分類"] = df["分類"].fillna("").replace(r'^\s*$', '未分類', regex=True)
    return df

def refresh_summary():
    try:
        df = _read_df()
        if df.empty:
            return "目前沒有資料", 0.0, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), None, None, None
        total = float(df["金額"].sum())
        by_cat, settle = _make_summary_tables(df)
        # Generate plots
        cat_pie_plot = _plot_category_pie_chart(df)
        monthly_line_plot = _plot_monthly_spending_line_chart(df)
        category_bar_plot = _plot_category_bar_chart(df)

        return "✅ 已更新彙總", total, by_cat, settle, df, cat_pie_plot, monthly_line_plot, category_bar_plot
    except Exception as e:
        return f"❌ 讀取失敗：{e}", 0.0, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), None, None, None

def _make_summary_tables(df: pd.DataFrame):
    # 分類小計
    if "分類" in df.columns:
        by_cat = df.groupby("分類", as_index=False)["金額"].sum().sort_values("金額", ascending=False)
    else:
        by_cat = pd.DataFrame(columns=["分類", "金額"])

    # 付款人結算（AA 分攤）
    if "付款人" in df.columns and df["付款人"].notna().any():
        payers = df["付款人"].replace("", "匿名")
        unique_payers = sorted([p for p in payers.unique() if p])
        total = df["金額"].sum()
        n = max(len(unique_payers), 1)
        aa_share = total / n
        paid_by = df.groupby("付款人", as_index=False)["金額"].sum().rename(columns={"金額": "實付"})
        # 確保每位付款人都有列
        all_rows = pd.DataFrame({"付款人": unique_payers}).merge(paid_by, on="付款人", how="left").fillna({"實付": 0.0})
        all_rows["應付(AA)"] = all_rows["實付"].sum() / len(all_rows) if len(all_rows) > 0 else 0.0 # Correct AA calculation
        all_rows["差額(實付-應付)"] = all_rows["實付"] - all_rows["應付(AA)"]
        settle = all_rows.sort_values("差額(實付-應付)", ascending=False)
    else:
        settle = pd.DataFrame(columns=["付款人", "實付", "應付(AA)", "差額(實付-應付)"])
    return by_cat, settle

def _plot_category_pie_chart(df: pd.DataFrame):
    if df.empty or "分類" not in df.columns or "月份" not in df.columns:
        return None

    # Filter out invalid dates/months if any
    df_valid = df.dropna(subset=['日期', '月份'])

    if df_valid.empty:
        return None

    # Group by month and category to get spending for each category per month
    monthly_category_spending = df_valid.groupby(['月份', '分類'])['金額'].sum().reset_index()

    # Get unique months to iterate and create plots for each month
    unique_months = sorted(df_valid['月份'].unique())

    if not unique_months:
        return None

    # Create a figure to hold all pie charts, one row per month
    num_months = len(unique_months)
    fig, axes = plt.subplots(num_months, 1, figsize=(8, 6 * num_months))
    # Ensure axes is an array even for a single subplot
    if num_months == 1:
        axes = [axes]

    for i, month in enumerate(unique_months):
        month_data = monthly_category_spending[monthly_category_spending['月份'] == month]
        if not month_data.empty:
            ax = axes[i]
            ax.pie(month_data['金額'], labels=month_data['分類'], autopct='%1.1f%%', startangle=90)
            ax.set_title(f'{month} - 各類別花費比例')
            ax.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.

    plt.tight_layout()
    return fig

def _plot_monthly_spending_line_chart(df: pd.DataFrame):
    if df.empty or "日期" not in df.columns or "月份" not in df.columns:
        return None

    # Filter out invalid dates/months if any
    df_valid = df.dropna(subset=['日期', '月份'])

    if df_valid.empty:
        return None

    # Group by month and sum the spending
    monthly_total_spending = df_valid.groupby('月份')['金額'].sum().reset_index()
    monthly_total_spending['月份'] = monthly_total_spending['月份'].astype(str)

    fig = plt.figure(figsize=(10, 6))
    sns.lineplot(x='月份', y='金額', data=monthly_total_spending, marker='o')
    plt.title('每月花費趨勢')
    plt.xlabel('月份')
    plt.ylabel('總金額')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    return fig

def _plot_category_bar_chart(df: pd.DataFrame):
    if df.empty or "分類" not in df.columns:
        return None

    # Group by category and sum the spending
    category_spending = df.groupby('分類')['金額'].sum().reset_index()
    category_spending = category_spending.sort_values(by='金額', ascending=False)

    fig = plt.figure(figsize=(10, 6))
    sns.barplot(x='分類', y='金額', data=category_spending, palette='viridis')
    plt.title('各類別花費比較')
    plt.xlabel('分類')
    plt.ylabel('總金額')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y')
    plt.tight_layout()
    return fig

In [9]:
with gr.Blocks(title="日常支出速算與分攤") as demo:
    gr.Markdown("## 🧾 日常支出速算與分攤（Gradio）\n- 新增支出後自動寫回 Google Sheet\n- 一鍵查看總額、分類小計與 AA 分攤\n- 讀寫工作表：`工作表1`")

    with gr.Tab("➕ 新增支出"):
        with gr.Row():
            date_in = gr.Textbox(label="日期 YYYY-MM-DD", value=datetime.date.today().strftime("%Y-%m-%d"))
            time_in = gr.Textbox(label="時間 HH:MM（可留白）", value="")
        with gr.Row():
            cat_in = gr.Textbox(label="分類", placeholder="如 外食 / 交通 / 購物")
            item_in = gr.Textbox(label="品項", placeholder="如 咖啡 / 便當 / 車票")
        with gr.Row():
            amt_in = gr.Textbox(label="金額", placeholder="數字")
            payer_in = gr.Textbox(label="付款人", placeholder="如 Pecu / Alice / Bob")
        with gr.Row():
            location_in = gr.Textbox(label="地點", placeholder="如 公司 / 家 / 餐廳")
            payment_method_in = gr.Textbox(label="支付方式", placeholder="如 現金 / 信用卡 / 行動支付")
        add_btn = gr.Button("新增到工作表")

        add_msg = gr.Markdown()
        total_out = gr.Number(label="目前總額", interactive=False)
        cat_df = gr.Dataframe(label="分類小計", interactive=False)
        settle_df = gr.Dataframe(label="AA 分攤結算", interactive=False)

        add_btn.click(
            fn=add_expense,
            inputs=[date_in, time_in, cat_in, item_in, amt_in, payer_in, location_in, payment_method_in],
            outputs=[add_msg, total_out, cat_df, settle_df]
        )

    with gr.Tab("📊 彙總 / AA 分攤"):
        refresh_btn = gr.Button("讀取最新彙總")
        msg2 = gr.Markdown()
        total2 = gr.Number(label="總額", interactive=False)
        cat_df2 = gr.Dataframe(label="分類小計", interactive=False)
        settle_df2 = gr.Dataframe(label="AA 分攤結算", interactive=False)
        raw_preview = gr.Dataframe(label="（預覽）最近資料", interactive=False)

        refresh_btn.click(
            fn=refresh_summary,
            inputs=[],
            outputs=[msg2, total2, cat_df2, settle_df2, raw_preview]
        )

    with gr.Tab("📈 圖表分析"):
        chart_refresh_btn = gr.Button("重新整理圖表")
        msg3 = gr.Markdown()
        category_pie_chart = gr.Plot(label="每月各類別花費比例")
        monthly_line_chart = gr.Plot(label="每月花費趨勢")
        category_bar_chart = gr.Plot(label="各類別花費比較")

        chart_refresh_btn.click(
            fn=refresh_summary, # Reusing refresh_summary to get updated plots
            inputs=[],
            outputs=[msg3, gr.State(), gr.State(), gr.State(), gr.State(), category_pie_chart, monthly_line_chart, category_bar_chart] # Only update plots
        )

    with gr.Tab("📒 檢視原始資料"):
        view_btn = gr.Button("讀取資料")
        view_df = gr.Dataframe(label="全部資料", interactive=False)

        def _view_all():
            try:
                df = _read_df()
                if df.empty:
                    return pd.DataFrame(columns=REQUIRED_COLUMNS)
                return df
            except Exception as e:
                return pd.DataFrame({"錯誤": [str(e)]})

        view_btn.click(fn=_view_all, inputs=[], outputs=[view_df])

# 啟動介面
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14768c2bd90c6fb97c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
